# 🧾 Receipt Text Extraction — Supermarket Optimizer

**Goal:** turn a supermarket receipt (in *any* form) into clean structured data:
`store · date · [ item · quantity · unit · price ]`

**Inputs come in three shapes** (increasing difficulty):

| # | File | Type | Difficulty | Why |
|---|------|------|-----------|-----|
| 1 | `Netto_Kassenbon_*.pdf` | App-exported **PDF** | ⭐ easy | has a real text layer — no OCR needed |
| 2 | `2026.06.30_*.jpg.png` | App-exported **PNG** | ⭐⭐ medium | clean, flat, high-contrast digital image |
| 3 | `IMG_1929.JPG` | **Photo** of a paper receipt | ⭐⭐⭐ hard | perspective, shadows, curl, OCR noise |

### Design constraints (from the product team)
The current prototype calls **Gemini Vision** for everything. That is accurate but has real problems:
- 🚦 **Rate limits** — the free tier throttles (429s); it does *not* "always work".
- 🔒 **Privacy** — receipts contain card PANs, transaction IDs, PAYBACK numbers. Uploading them to a third party is a GDPR liability.
- 💸 **Cost / dependency** — every scan is a paid, network-dependent API call.

### The strategy in this notebook: **local-first, tiered pipeline**
Do as much as possible **on-device**, with **no network and no rate limits**:

```
                 ┌── PDF with text layer ──►  extract text directly      (perfect, instant, 100% local)
 receipt file ──►┤── PDF scanned / image  ──►  Tesseract OCR + preprocess (local, no limits)
                 └── parse text ──► regex extractor ──► {store, date, items}
                                        │
                                        └── (optional) LLM *cleanup* only when confidence is low
```

The cloud/LLM step is **demoted to an optional fallback** — used only to clean up garbled photo OCR, never as the default. This satisfies all three constraints: no rate limits, always works offline, and raw receipts never leave the machine.


## What I evaluated and why

| Approach | Rate limits | Privacy | Accuracy | Cost | Verdict |
|---|---|---|---|---|---|
| **Cloud Vision LLM** (Gemini / Claude) | ❌ throttled | ❌ data leaves device | ✅ excellent | 💸 per-call | keep as *optional fallback* |
| **PDF text-layer** (PyMuPDF / pdfplumber) | ✅ none | ✅ fully local | ✅✅ exact chars | free | **primary for PDFs** |
| **Local OCR** (Tesseract `deu`) | ✅ none | ✅ fully local | ✅ great on clean, decent on photos | free | **primary for images** |
| Local OCR alt (EasyOCR / PaddleOCR) | ✅ none | ✅ fully local | ✅ better on photos, heavier install | free | good upgrade path |
| Local LLM cleanup (Ollama) | ✅ none | ✅ fully local | ✅ fixes garbled names | free | privacy-safe alternative to cloud |

**Chosen stack:** PyMuPDF (PDF) + Tesseract-OCR German + OpenCV preprocessing + a rule-based parser. Everything below runs offline.


In [1]:
# ── Setup ───────────────────────────────────────────────────────────────────
# One-time install (local, free, no API keys):
#   pip install pymupdf pytesseract pillow opencv-python-headless pandas
#   brew install tesseract                      # the OCR engine
#   # German language model (best-quality "tessdata_best"):
#   curl -L -o "$(brew --prefix)/share/tessdata/deu.traineddata" \
#     https://github.com/tesseract-ocr/tessdata_best/raw/main/deu.traineddata
#
# Optional (only for the fallback in the last section): pip install google-genai python-dotenv

import re
import fitz                      # PyMuPDF — PDF text layer + rendering
import cv2                       # OpenCV — image preprocessing
import numpy as np
import pandas as pd
import pytesseract               # Tesseract OCR wrapper
from PIL import Image, ImageOps
from pathlib import Path

RECEIPTS = Path("receipts")

# Sanity check: is the German OCR model available?
langs = pytesseract.get_languages(config="")
print("Tesseract languages available:", langs)
assert "deu" in langs, "German model missing — see install note above (deu.traineddata)"
print("✅ Ready — German OCR model found.")

Tesseract languages available: ['deu', 'eng', 'osd', 'snum']
✅ Ready — German OCR model found.


## Step 1 — Text extraction (the two local engines)

**Engine A — PDF text layer.** App-exported receipts (Netto eBon, many Rewe/Edeka app PDFs) embed the *actual characters*. We read them directly — this is exact, instant, and fully local. We only fall back to OCR if the PDF has no text layer (i.e. it's a scanned image inside a PDF).

**Engine B — Image OCR.** For PNG/JPG we run Tesseract. Two things matter a lot for German receipts:
1. **The `deu` language model** — receipts are full of `ä ö ü ß` and German words.
2. **Preprocessing** — fix EXIF rotation, upscale, greyscale, and (for photos) adaptive-threshold to a clean black-on-white image. This is what lifts a noisy phone photo into something OCR-able.


In [2]:
# ── Engine A: PDF text layer (with OCR fallback for scanned PDFs) ────────────
def extract_pdf_text(path):
    """Return (text, used_ocr). Uses the embedded text layer when present."""
    doc = fitz.open(path)
    text = "\n".join(page.get_text() for page in doc)
    if text.strip():
        return text, False
    # No text layer → it's a scanned PDF. Render pages to images and OCR them.
    pages = []
    for page in doc:
        pix = page.get_pixmap(dpi=300)
        img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
        pages.append(ocr_image_pil(img, photo=False))
    return "\n".join(pages), True


# ── Engine B: image OCR with preprocessing ──────────────────────────────────
def preprocess(path_or_pil, photo=True):
    """Load, fix orientation, greyscale, upscale, and (for photos) binarise."""
    img = path_or_pil if isinstance(path_or_pil, Image.Image) else Image.open(path_or_pil)
    img = ImageOps.exif_transpose(img).convert("RGB")           # honour phone rotation
    g = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2GRAY)
    scale = 2000 / g.shape[1]                                   # upscale to ~2000px wide
    if scale > 1:
        g = cv2.resize(g, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    if photo:                                                   # photos need denoise + threshold
        g = cv2.bilateralFilter(g, 7, 50, 50)
        g = cv2.adaptiveThreshold(g, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                  cv2.THRESH_BINARY, 31, 15)
    return Image.fromarray(g)

def ocr_image_pil(pil_img, photo=True):
    prepared = preprocess(pil_img, photo=photo)
    # --oem 1 = LSTM engine, --psm 6 = assume a single uniform block of text
    return pytesseract.image_to_string(prepared, lang="deu", config="--oem 1 --psm 6")

def ocr_image(path, photo=True):
    return ocr_image_pil(Image.open(path), photo=photo)

print("Extraction engines defined: extract_pdf_text() + ocr_image()")

Extraction engines defined: extract_pdf_text() + ocr_image()


## Step 2 — One entry point + automatic layout detection

`receipt_to_text(path)` routes by file type so the caller doesn't care whether it's a PDF or a photo.

It also **auto-detects the receipt layout**, because German receipts come in two shapes that need different parsers:

- **Inline** (printed/OCR'd tills — Lidl, Edeka): `Banane Fair Bio      2,35 A`  ← name and price on one line.
- **Block** (digital eBon PDFs — Netto): the name is on one line and the price on the *next* line.

We decide by counting how many lines are a lone price vs. a name-then-price line.


In [3]:
def detect_layout(text):
    """'block' = price on its own line (eBon PDFs); 'inline' = price at end of item line."""
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    lone   = sum(1 for l in lines if re.match(r"^-?\d{1,3},\d{2}$", l))
    inline = sum(1 for l in lines if re.search(r"[A-Za-zÄÖÜäöü].*\d,\d{2}\s*[A-Za-z]?\s*$", l))
    return "block" if lone >= 3 and lone >= inline else "inline"

def receipt_to_text(path):
    """Route any receipt file → (raw_text, source_kind, layout)."""
    path = Path(path)
    ext = path.suffix.lower()
    if ext == ".pdf":
        text, used_ocr = extract_pdf_text(path)
        kind = "pdf-ocr" if used_ocr else "pdf-text"
    else:
        # heuristic: a huge phone photo needs the photo pipeline; a small flat
        # app-export (PNG / screenshot) is already clean.
        w, h = Image.open(path).size
        is_photo = ext in (".jpg", ".jpeg") and max(w, h) > 2500
        text = ocr_image(path, photo=is_photo)
        kind = "image-photo" if is_photo else "image-clean"
    return text, kind, detect_layout(text)

print("Router defined: receipt_to_text(path) → (text, kind, layout)")

Router defined: receipt_to_text(path) → (text, kind, layout)


## Step 3 — Parse the text into structured fields

Now we turn raw text into `{store, date, items[]}`. This is pure rules (regex) — fast, free, deterministic, and easy to audit. Key jobs:

- **Store** — match known chains in the header region.
- **Date** — first `dd.mm.yy(yy)` on the receipt, normalised to ISO `YYYY-MM-DD`.
- **Items** — name + price, plus:
  - **quantity multipliers** — `Puten-Geschnetzeltes 4,59 x 2` → qty 2.
  - **weighed goods** — `1,266 kg x 1,29 EUR/kg` → quantity 1.266, unit `kg`.
- **Noise filtering** — drop totals, tax tables, PAYBACK, TSE signatures, card/payment lines, discounts. This is also *privacy-positive*: card PANs and transaction IDs never make it into the output.


In [4]:
# ── Known German chains (header match) ──────────────────────────────────────
STORES = ["Lidl","Rewe","E-Center","E center","Edeka","Aldi","Netto","Norma",
          "Penny","Kaufland","Real","dm","Rossmann"]

# Lines that are NOT products → skipped (continue)
NOISE = re.compile(
    r"mwst|brutto|^netto$|geg\.|^bar\b|kartenzahlung|kreditkarte|ec-?karte|girocard|"
    r"^visa|^master|pfand|rückgeld|payback|coupon|rabatt|preisvorteil|posten|tse|"
    r"prüfwert|seriennr|terminal|genehmigung|steuer|kundenbeleg|gespart|punkte|"
    r"bon-nr|^\s*[a-z]\s*\d+%|^\s*\d+%\s", re.I)

# Lines that mark the totals block → stop reading items (break)
TOTAL = re.compile(r"\bsumme\b|zu\s*zahlen|gesamtbetrag|\bgesamt\b|zu\s*zahl", re.I)

PRICE_END   = re.compile(r"(-?\d{1,4},\d{2})\s*[A-Za-z]{0,3}\s*$")   # inline: price at line end
PRICE_ONLY  = re.compile(r"^-?\d{1,3},\d{2}$")                        # block: lone price line
WEIGHT_X    = re.compile(r"(\d+[.,]\d+)\s*(kg|g|l|ml|stk)\s*x", re.I) # "1,266 kg x"
UNIT_PRICE  = re.compile(r"eur\s*/\s*(kg|g|l|stk)", re.I)             # "1,29 EUR/kg"
QTY_INLINE  = re.compile(r"x\s*%?\s*(\d+)")                           # "... x 2"
DATE_RE     = re.compile(r"(\d{2})\.(\d{2})\.(\d{2,4})")

def find_store(text):
    for s in STORES:
        if re.search(re.escape(s), text[:400], re.I):
            return "Edeka" if s.lower().replace("-", " ") == "e center" else s
    return "unknown"

def find_date(text):
    m = DATE_RE.search(text)
    if not m:
        return None
    d, mth, y = m.groups()
    y = ("20" + y) if len(y) == 2 else y            # 26 → 2026
    return f"{y}-{mth}-{d}"                          # ISO YYYY-MM-DD

def clean_name(name):
    name = re.sub(r"^[^0-9A-Za-zÄÖÜäöüß]+", "", name)   # strip leading OCR junk (‚ / | _ …)
    return re.sub(r"\s{2,}", " ", name).strip(" .:-*|")

def is_valid_name(name):
    return len(name) >= 3 and not name.replace(" ", "").isdigit()

print("Parsing helpers defined.")

Parsing helpers defined.


In [5]:
# ── Parser A: BLOCK layout (digital eBon PDFs) ──────────────────────────────
def parse_block(text):
    items, name, weight = [], None, None
    for raw in text.splitlines():
        l = raw.strip()
        if re.match(r"summe", l, re.I):        # totals reached
            break
        if not l:
            continue
        wm = WEIGHT_X.match(l)
        if wm:                                 # remember the weight for the coming item
            weight = (float(wm.group(1).replace(",", ".")), wm.group(2).lower()); continue
        if UNIT_PRICE.search(l):               # "1,29 EUR/kg" — skip the unit-price line
            continue
        if PRICE_ONLY.match(l):                # a lone price closes the pending item
            if name:
                p = float(l.replace(",", "."))
                if p > 0:
                    q, u = weight if weight else (1, "Stk")
                    items.append({"name": name, "quantity": q, "unit": u, "price_eur": p})
                name, weight = None, None
            continue
        if len(l) <= 1 or l in ("B", "A", "EUR", "***"):
            continue
        name = l                               # otherwise this line is the item name
    return items

# ── Parser B: INLINE layout (OCR'd till receipts) ───────────────────────────
def parse_inline(text):
    items = []
    for raw in text.splitlines():
        line = raw.strip()
        if not line:
            continue
        wm = WEIGHT_X.search(line)             # weight line refers to the PREVIOUS item
        if wm and "eur/" in line.lower().replace(" ", ""):
            if items and items[-1]["unit"] == "Stk":
                items[-1]["quantity"] = float(wm.group(1).replace(",", "."))
                items[-1]["unit"] = wm.group(2).lower()
            continue
        if TOTAL.search(line):                 # reached totals/payment → stop
            break
        if NOISE.search(line):
            continue
        m = PRICE_END.search(line)
        if not m:
            continue
        price = float(m.group(1).replace(",", "."))
        if price <= 0:                         # discount / coupon line
            continue
        name = line[:m.start()]
        qty, unit = 1, "Stk"
        qm = QTY_INLINE.search(name)           # "4,59 x 2"
        if qm:
            qty = int(qm.group(1))
            name = re.split(r"\d+[.,]\d{2}\s*€?\s*x", name)[0]
        name = clean_name(name)
        if not is_valid_name(name):
            continue
        items.append({"name": name, "quantity": qty, "unit": unit, "price_eur": price})
    return items

def parse_items(text, layout):
    return parse_block(text) if layout == "block" else parse_inline(text)

print("Parsers defined: parse_block / parse_inline")

Parsers defined: parse_block / parse_inline


## Step 4 — The full pipeline + a built-in confidence check

`extract_receipt(path)` chains everything: route → OCR/text → parse → assemble.

As a **data-quality signal** it also reconciles the sum of item prices against the receipt's printed total (`SUMME`). If they match, we're confident the parse is complete; if not, the receipt is flagged for review (or for the optional LLM fallback in the last section).


In [6]:
SUM_RE = re.compile(r"(?:summe|zu\s*zahlen|gesamt)\D{0,15}(\d{1,4},\d{2})", re.I)

def printed_total(text):
    hits = SUM_RE.findall(text)
    return float(hits[0].replace(",", ".")) if hits else None

def extract_receipt(path):
    text, kind, layout = receipt_to_text(path)
    items = parse_items(text, layout)
    # each price_eur is already the LINE total (qty is metadata), so just add them up
    subtotal = round(sum(i["price_eur"] for i in items), 2)
    total = printed_total(text)
    # confidence: does the parsed subtotal reconcile with the printed total?
    if total is None:
        confidence = "unknown"
    elif abs(subtotal - total) < 0.02:
        confidence = "high"
    elif abs(subtotal - total) / total < 0.15:
        confidence = "medium"
    else:
        confidence = "low"
    return {
        "file": Path(path).name,
        "source": kind,
        "layout": layout,
        "store": find_store(text),
        "date": find_date(text),
        "item_count": len(items),
        "parsed_subtotal": subtotal,
        "printed_total": total,
        "confidence": confidence,
        "items": items,
        "_raw": text,          # kept for inspection; drop in production
    }

print("Pipeline defined: extract_receipt(path)")

Pipeline defined: extract_receipt(path)


## Step 5 — Run it on all three receipts

One function, three very different inputs, zero network calls.


In [7]:
TARGETS = [
    RECEIPTS / "Netto_Kassenbon_20260707-180115.pdf",       # ⭐ PDF (text layer)
    RECEIPTS / "2026.06.30_23001961862026063034796.jpg.png",# ⭐⭐ clean PNG
    RECEIPTS / "IMG_1929.JPG",                              # ⭐⭐⭐ phone photo
]

results = [extract_receipt(p) for p in TARGETS]

for r in results:
    print("═" * 74)
    print(f"📄 {r['file']}")
    print(f"   source={r['source']:<11} layout={r['layout']:<6} store={r['store']:<8} date={r['date']}")
    print(f"   items={r['item_count']:<3} parsed={r['parsed_subtotal']}  printed_total={r['printed_total']}  confidence={r['confidence'].upper()}")
    print("   " + "-" * 68)
    for it in r["items"]:
        qty = f"{it['quantity']:g}"
        print(f"   {it['name'][:38]:<38} {qty:>6} {it['unit']:<4} {it['price_eur']:>7.2f} €")
print("═" * 74)

══════════════════════════════════════════════════════════════════════════
📄 Netto_Kassenbon_20260707-180115.pdf
   source=pdf-text    layout=block  store=Netto    date=2026-07-07
   items=10  parsed=14.15  printed_total=14.15  confidence=HIGH
   --------------------------------------------------------------------
   Beste Ernte Kichererbsen 265g               1 Stk     0.59 €
   Beste Ernte Kidneybohnen 255g               1 Stk     0.69 €
   Irische Butter Kerrygold 250g               1 Stk     1.49 €
   Bio BB Feta 200g                            1 Stk     2.69 €
   Bio Paprika Mix 400g                        1 Stk     1.99 €
   Minigurken 750g                             1 Stk     1.99 €
   Minipflaumen Tomaten 500g                   1 Stk     1.99 €
   Bananen Lose MT                         1.266 kg      1.63 €
   Lauchzwiebeln                               1 Stk     0.55 €
   Apfel Pink Lady Lose                    0.182 kg      0.54 €
════════════════════════════════════════════

### Same results as tidy tables (ready for a DataFrame / DB / CSV)

In [8]:
# Summary table — one row per receipt
summary = pd.DataFrame([{k: r[k] for k in
    ["file","source","layout","store","date","item_count",
     "parsed_subtotal","printed_total","confidence"]} for r in results])
display(summary)

# Long/tidy item table — one row per purchased item (this is what feeds the app DB)
rows = []
for r in results:
    for it in r["items"]:
        rows.append({"store": r["store"], "date": r["date"], **it})
items_df = pd.DataFrame(rows)
display(items_df)

# items_df.to_csv("extracted_items.csv", index=False)   # ← export when needed

,file,source,layout,store,date,item_count,parsed_subtotal,printed_total,confidence
0,Netto_Kassenbon_20260707-180115.pdf,pdf-text,block,Netto,2026-07-07,10,14.15,14.15,high
1,2026.06.30_23001961862026063034796.jpg.png,image-clean,inline,Lidl,2026-06-30,15,41.51,41.30,medium
2,IMG_1929.JPG,image-photo,inline,Edeka,NaN,8,17.06,165.46,low


,store,date,name,quantity,unit,price_eur
0,Netto,2026-07-07,Beste Ernte Kichererbsen 265g,1.000,Stk,0.59
1,Netto,2026-07-07,Beste Ernte Kidneybohnen 255g,1.000,Stk,0.69
2,Netto,2026-07-07,Irische Butter Kerrygold 250g,1.000,Stk,1.49
3,Netto,2026-07-07,Bio BB Feta 200g,1.000,Stk,2.69
4,Netto,2026-07-07,Bio Paprika Mix 400g,1.000,Stk,1.99
5,Netto,2026-07-07,Minigurken 750g,1.000,Stk,1.99
6,Netto,2026-07-07,Minipflaumen Tomaten 500g,1.000,Stk,1.99
7,Netto,2026-07-07,Bananen Lose MT,1.266,kg,1.63
8,Netto,2026-07-07,Lauchzwiebeln,1.000,Stk,0.55
9,Netto,2026-07-07,Apfel Pink Lady Lose,0.182,kg,0.54


## Step 6 (optional) — LLM *cleanup* fallback for hard photos

Look at the results above: **PDF and PNG are essentially perfect**, but the **photo** has garbled product names (`THUER,METT/HACKEP` should read *Thüringer Mett/Hack*, `Frischfi.erm.BED` etc.) — the classic limit of local OCR on a curled, shadowed phone photo.

**When to reach for an LLM (and when not to):**
- ✅ Only when local confidence is **low/medium** — so 90%+ of scans (PDFs, app PNGs) never touch the network → **no rate-limit pressure, no privacy exposure**.
- 🔒 **Redact before sending.** The parser already stripped card PANs, TSE signatures and PAYBACK numbers. We send only the *item lines*, never the raw receipt — so transaction data never leaves the device.
- 🏠 **Fully-private option:** run the same cleanup on a **local LLM via Ollama** (e.g. `llama3.1`) — no data leaves the machine and there are no rate limits at all. Same prompt, swap the client.

The cell below is **guarded** (`USE_LLM_FALLBACK = False`) so this notebook stays offline by default. Flip it to `True` to try Gemini on the low-confidence receipts only.


In [9]:
USE_LLM_FALLBACK = False   # ← set True to enable the cloud fallback (needs GEMINI_API_KEY)

def llm_cleanup(items, store):
    """Send ONLY item name+price (no card/transaction data) to an LLM to fix
    garbled German product names. Privacy-preserving: raw receipt is never sent."""
    from google import genai
    from google.genai import types
    from dotenv import load_dotenv
    load_dotenv()
    client = genai.Client()   # reads GEMINI_API_KEY

    payload = [{"name": i["name"], "price_eur": i["price_eur"]} for i in items]
    prompt = (
        f"These noisy OCR lines are food items from a German {store} receipt. "
        "Return JSON list with the same length; for each, fix the German product "
        "name (expand abbreviations, correct OCR errors) and keep the price. "
        "Do NOT invent or drop items.\n\n" + json.dumps(payload, ensure_ascii=False)
    )
    resp = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(response_mime_type="application/json",
                                           temperature=0.1),
    )
    return json.loads(resp.text)

import json
if USE_LLM_FALLBACK:
    for r in results:
        if r["confidence"] in ("low", "medium"):
            print(f"🔧 LLM cleanup for {r['file']} (confidence={r['confidence']}) …")
            try:
                cleaned = llm_cleanup(r["items"], r["store"])
                for orig, fix in zip(r["items"], cleaned):
                    print(f"   {orig['name']:<32} →  {fix.get('name')}")
            except Exception as e:
                print(f"   ⚠️ fallback unavailable ({e}) — keeping local result")
else:
    print("LLM fallback disabled. Local pipeline ran fully offline with no rate limits. ✅")
    low = [r["file"] for r in results if r["confidence"] in ("low", "medium", "unknown")]
    print("Receipts that WOULD be sent for cleanup (low confidence only):", low or "none")

LLM fallback disabled. Local pipeline ran fully offline with no rate limits. ✅
Receipts that WOULD be sent for cleanup (low confidence only): ['2026.06.30_23001961862026063034796.jpg.png', 'IMG_1929.JPG']


## Summary & recommendation

**What works best, per input type:**

| Input | Engine | Result | Network |
|---|---|---|---|
| App PDF (Netto) | PyMuPDF text layer | 🟢 exact — all items, weights, prices | none |
| App PNG (Lidl) | Tesseract `deu` + upscale | 🟢 all items correct | none |
| Phone photo (Edeka) | Tesseract `deu` + preprocess | 🟡 items & prices OK, some names garbled | none |

**Recommendation for the product:**

1. **Make the local pipeline the default.** It has **no rate limits, works offline, and keeps raw receipts on-device** — directly fixing the three problems with the Gemini-only prototype. PDFs and app-exported images (the majority of real usage) come out essentially perfect.
2. **Use the confidence check** (`parsed_subtotal` vs `printed_total`) to *automatically* decide when a scan is good enough.
3. **Only escalate low-confidence photos** to an LLM — and prefer a **local LLM (Ollama)** or a **redacted** cloud call (item lines only, never card/transaction data).
4. **Upgrade path for photos:** if photo accuracy needs to be higher without an LLM, swap Tesseract for **PaddleOCR/EasyOCR** (better on curved, low-contrast images) — still 100% local.

**Next steps:** perspective-correction (deskew/dewarp) for photos, a small store→product-name dictionary for common abbreviations, and wiring `items_df` into the app's database.
